# Tools
tools extends what agents can do. letting agent to fetch real-time data, execute code, query external databases, and take actions in the world.

In [ ]:
from langchain.tools import tool

#basic tool that takes a string and returns a string
@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

#tool with custom name and description
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

## tool with schema defination
define complex input with pydantic models or json schemas

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## Access context
Tools can access runtime information like conversation history, user data, and persistent memory using ToolRuntime parameter
- It provides:
    - State:
        - Short-term memory - mutable data that exists for the current conversation (messages, counters, custom fields)
        - Access conversation history, track tool call counts
    - Context:
        - Immutable configuration passed at invocation time (user IDs, session info)
        - Personalize responses based on user identity
    - Store:
        - Long-term memory - persistent data that survives across conversations
        - Save user preferences, maintain knowledge base
    - Stream Writer:
        - Emit real-time updates during tool execution
        - Show progress for long-running operations
    - Server Info:
        - Server-specific metadata when running on LangGraph Server (assistant ID, graph ID, authenticated user)
        - Access assistant ID, graph ID, or authenticated user info
    - Execution Info:
        - Identity and retry information for the current execution (thread ID, run ID, attempt number)
        - Access thread/run IDs, adjust behavior based on retry state
    - Config:
        - RunnableConfig for the execution
        - Access callbacks, tags, and metadata
    - Tool Call ID:	
        - Unique identifier for the current tool invocation
        - Correlate tool calls for logs and model invocations

ToolRuntime is automatically injected and hidden from LLM - it wont appear in the tool's schema

### State: Short-term memory
State represents short-term memory that exists for the duration of a conversation. 

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]

    # Find the last human message
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

    return "No user messages found"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")

#### update state
This is useful for tools that need to update custom state fields. 
Use Command to update the agent’s state.
Include a ToolMessage in the update so the model can see the result of the tool call

In [ ]:
from langchain.agents import AgentState
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


class CustomState(AgentState):
    user_name: str


@tool
def set_user_name(new_name: str, runtime: ToolRuntime[None, CustomState]) -> Command:
    """Set the user's name in the conversation state."""
    return Command(
        update={
            "user_name": new_name,
            "messages": [
                ToolMessage(
                    content=f"User name set to {new_name}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

#### Access state
Tools can access the current conversation state using runtime.state

### Context
Context provides immutable configuration data that is passed at invocation time. Use it for user IDs, session details, or application-specific settings that shouldn’t change during a conversation.
- Access context through runtime.context. Pass it alongside a thread_id so the conversation is persisted across turns:

In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.utils.uuid import uuid7
from langchain_openai import ChatOpenAI

USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}


@dataclass
class UserContext:
    user_id: str


def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"Account holder: {user['name']}\n"
            f"Type: {user['account_type']}\n"
            f"Balance: ${user['balance']}"
        )
    return "User not found"

model = ChatOpenAI(model="google_genai:gemini-3.5-flash")
agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
    context=UserContext(user_id="user123"),
)


### Store: Long-Term memory (Store)
The BaseStore provides persistent storage that survives across conversations.

In [ ]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI

# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Successfully saved user info."

model = ChatOpenAI(model="gpt-5.5")

store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
# Here is the user info for user with ID "abc123":
# - Name: Foo
# - Age: 25
# - Email: foo@langchain.dev

### Stream Writer
This is useful for providing progress feedback to users during long-running operations.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city."""
    writer = runtime.stream_writer

    # Stream custom updates as the tool executes
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")

    return f"It's always sunny in {city}!"

### Execution Info
Access thread ID, run ID, and retry state from within a tool via runtime.execution_info

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def log_execution_context(runtime: ToolRuntime) -> str:
    """Log execution identity information."""
    info = runtime.execution_info
    print(f"Thread: {info.thread_id}, Run: {info.run_id}")
    print(f"Attempt: {info.node_attempt}")
    return "done"

### Config

## Tool return values
You can choose different return values for your tools:
- Return a string for human-readable results.
- Return an object for structured results the model should parse.
- Return a Command with optional message when you need to write to state.


### string Return
Return a string when the tool should provide plain text for the model to read and use in its next response.
- Behavior

    - The return value is converted to a ToolMessage.
    - The model sees that text and decides what to do next.
    - No agent state fields are changed unless the model or another tool does so later.


In [ ]:
from langchain.tools import tool
@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is currently sunny in {city}."

### Return an Object
Return an object (for example, a dict) when your tool produces structured data that the model should inspect.
- Behavior:
    - The object is serialized and sent back as tool output.
    - The model can read specific fields and reason over them.
    - Like string returns, this does not directly update graph state.


In [ ]:
from langchain.tools import tool


@tool
def get_weather_data(city: str) -> dict:
    """Get structured weather data for a city."""
    return {
        "city": city,
        "temperature_c": 22,
        "conditions": "sunny",
    }

### Return a Command
Return a Command when the tool needs to update graph state (for example, setting user preferences or app state).
- Behavior:
    - The command updates state using update.
    - Updated state is available to subsequent steps in the same run.
    - Use reducers for fields that may be updated by parallel tool calls.
    
Use this when the tool is not just returning data, but also mutating agent state.

In [ ]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


@tool
def set_language(language: str, runtime: ToolRuntime) -> Command:
    """Set the preferred response language."""
    return Command(
        update={
            "preferred_language": language,
            "messages": [
                ToolMessage(
                    content=f"Language set to {language}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

## Return Directly from tool
the agent returns the tool’s output to the caller immediately, without sending it back through the model for further processing.
- Behavior:
    - The tool executes normally and its output is wrapped in a ToolMessage.
    - The agent stops looping and returns the tool’s output as the final response, bypassing any additional model call.
    - If the model calls multiple tools in a single turn, return_direct takes effect only when all called tools have return_direct=True.
- Use this when:
    - The tool’s output is the complete, user-ready answer (for example, a lookup that returns a ready-to-display result).
    - You want to avoid an extra model call when no additional reasoning is needed.
    - You need deterministic, unmodified output — the model cannot rephrase, summarize, or act on the tool result.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI


@tool(return_direct=True)
def fetch_order_status(order_id: str) -> str:
    """Fetch the current status of a customer order."""
    # In production, query your order management system here
    return f"Order {order_id} is shipped and will arrive in 2 days."


agent = create_agent(
    ChatOpenAI(model="google_genai:gemini-3.5-flash"),
    tools=[fetch_order_status],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "What is the status of order #12345?"}]
})
# The agent returns the tool output directly without another LLM call:
# "Order 12345 is shipped and will arrive in 2 days."

## Error Handling
Handle tool errors using LangChain agent middleware to retry failed tool calls or return custom error messages:

In [ ]:
from collections.abc import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest


@wrap_tool_call
def handle_tool_errors(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage],
) -> ToolMessage:
    """Convert tool exceptions into ToolMessages the model can handle."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({e})",
            tool_call_id=request.tool_call["id"],
        )


agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[],
    middleware=[handle_tool_errors],
)